# Puppies of the Dow - Research Notebook

**Source**: QC Strategy Library - Dogs/Puppies of the Dow

**Concept**: A value-oriented strategy selecting the 5 lowest-priced stocks from the 10 highest-yielding Dow Jones components. Annual rebalance at year start. The "puppies" variant adds a low-price filter on top of the classic "Dogs of the Dow" high-yield screen.

In [1]:
try:
    from AlgorithmImports import *
    qb = QuantBook()
    QC_ENV = True
except (ImportError, NameError):
    import pandas as pd
    import numpy as np
    from datetime import datetime, timedelta
    QC_ENV = False
    qb = None

if QC_ENV:
    qb.AddEquity("DIA", Resolution.Daily)
    qb.AddEquity("SPY", Resolution.Daily)
    print("Universe: Dow Jones Industrial Average (30 stocks via DIA)")
    print("Selection: Top 10 by yield -> Top 5 by lowest price")
else:
    print("Local environment - loading Dow components + SPY benchmark via yfinance")

Local environment - loading Dow components + SPY benchmark via yfinance


## Strategy Logic

1. **Universe**: DIA ETF holdings (Dow Jones 30 components)
2. **Filter**: Price > $5 AND total yield > 0
3. **Dogs**: Top 10 by total yield (dividend + buyback yield)
4. **Puppies**: Top 5 lowest-priced from the Dogs
5. **Weighting**: Equal weight (20% each)
6. **Rebalance**: Annual at year start

In [2]:
# Fetch SPY benchmark data - handle QC Cloud MemoizingEnumerable and local yfinance fallback
if QC_ENV:
    spy_raw = qb.History("SPY", timedelta(252 * 12), Resolution.Daily)
    if isinstance(spy_raw, pd.DataFrame):
        spy_hist = spy_raw
    else:
        spy_hist = pd.DataFrame([{'close': float(b.Close), 'open': float(b.Open),
                                   'high': float(b.High), 'low': float(b.Low),
                                   'volume': float(b.Volume)} for b in spy_raw])
else:
    import yfinance as yf
    spy_hist = yf.Ticker("SPY").history(period="12y", auto_adjust=True).reset_index()
    spy_hist.columns = [str(c).lower() for c in spy_hist.columns]

spy_returns = spy_hist["close"].pct_change().dropna()

print(f"SPY 12-year data: {len(spy_hist)} trading days")
print(f"  Annualized return: {(1 + spy_returns.mean())**252 - 1:.2%}")
print(f"  Annualized vol: {spy_returns.std() * (252**0.5):.2%}")
print(f"  Sharpe (rf=0): {(spy_returns.mean() / spy_returns.std()) * (252**0.5):.2f}")

SPY 12-year data: 3017 trading days
  Annualized return: 15.67%
  Annualized vol: 17.47%
  Sharpe (rf=0): 0.83


## The screen, executed for real

The strategy's entire subject is the selection: **top 10 Dow components by
dividend yield -> the 5 lowest-priced of those ("the puppies") -> equal
weight**. Below it runs on current market data via yfinance.

Honest scope note: the QC strategy screens on **total yield** (dividends +
buybacks) from DIA holdings; buyback yield is not in yfinance, so the local
screen uses the dividend yield -- the measurable proxy -- and says so in the
output.

In [3]:
# REAL Dogs -> Puppies selection on current data (no simulation)
import yfinance as yf
import pandas as pd

# Current DJIA composition (static list as of writing; the QC engine reads live DIA holdings)
DOW_30 = [
    "AAPL", "AMGN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DD", "DIS",
    "DOW", "GS", "HD", "HON", "IBM", "JPM", "KO", "MCD", "MMM", "MRK",
    "MSFT", "NVDA", "NKE", "PFE", "PG", "TRV", "UNH", "V", "VZ", "WMT",
]

rows = {}
for tk in DOW_30:
    try:
        t = yf.Ticker(tk)
        info = t.info
        dy = info.get("dividendYield")
        px = info.get("currentPrice") or info.get("regularMarketPrice")
        if dy is None or px is None:
            dy2 = info.get("trailingAnnualDividendYield")
            if dy is None and dy2 is not None:
                dy = dy2 * 100
        if dy is not None and px is not None:
            rows[tk] = {"price": float(px), "div_yield_pct": float(dy)}
    except Exception:
        pass

dow = pd.DataFrame(rows).T
print(f"Dow 30 data loaded: {len(dow)}/30 components (price + dividend yield)\n")

# Step 1-2: filter price > $5 and yield > 0
eligible = dow[(dow["price"] > 5) & (dow["div_yield_pct"] > 0)].copy()
print(f"Eligible (price > $5, yield > 0): {len(eligible)}/30")

# Step 3: DOGS = top 10 by dividend yield
dogs = eligible.sort_values("div_yield_pct", ascending=False).head(10)
print(f"\nDOGS OF THE DOW (top 10 by dividend yield, measured now):")
print(dogs.sort_values("div_yield_pct", ascending=False).to_string())

# Step 4: PUPPIES = 5 lowest-priced of the dogs
puppies = dogs.sort_values("price").head(5)
print(f"\nPUPPIES OF THE DOW (5 lowest-priced among the dogs -- the portfolio):")
print(puppies.to_string())
print(f"\nEqual-weighted allocation: {', '.join(f'{tk} 20%' for tk in puppies.index)}")

Dow 30 data loaded: 30/30 components (price + dividend yield)

Eligible (price > $5, yield > 0): 29/30

DOGS OF THE DOW (top 10 by dividend yield, measured now):
      price  div_yield_pct
VZ    47.08           6.01
PFE   28.67           6.00
DOW   28.02           5.00
NKE   35.75           4.59
CVX  204.45           3.48
MCD  236.50           3.26
HD   293.20           3.18
IBM  225.51           3.00
PG   146.23           2.98
UNH  376.59           2.46

PUPPIES OF THE DOW (5 lowest-priced among the dogs -- the portfolio):
      price  div_yield_pct
DOW   28.02           5.00
PFE   28.67           6.00
NKE   35.75           4.59
VZ    47.08           6.01
PG   146.23           2.98

Equal-weighted allocation: DOW 20%, PFE 20%, NKE 20%, VZ 20%, PG 20%


## Key Observations

- **Dogs of the Dow** is a classic value strategy dating back to the 1990s
- **Puppies variant** adds a low-price filter, selecting for deeper undervaluation
- **Annual rebalance** keeps turnover very low (one trade per year)
- **Concentrated portfolio** (5 stocks) has higher idiosyncratic risk
- **The screen above is measured on live data**; the 12y backtest requires
  historical Dow membership per rebalance date and runs on QC Cloud -- today's
  composition projected backward would be survivorship bias

In [4]:
# What is measured here vs what the 12y backtest requires -- stated honestly
print("=" * 58)
print("MEASURED HERE (current snapshot, real data)")
print("=" * 58)
print(f"  Dow components screened: {len(dow)}/30")
print(f"  Dogs (top 10 by div yield): {', '.join(dogs.index)}")
print(f"  Puppies (5 lowest-priced): {', '.join(puppies.index)}")
print()
print("=" * 58)
print("NOT MEASURED HERE: the 12-year annual-rebalance backtest")
print("=" * 58)
print("A faithful backtest needs the Dow composition AND each year's yield")
print("as of each rebalance date -- historical index membership. Applying")
print("TODAY's puppies to the past would be survivorship bias, so this")
print("notebook does not fake it.")
print()
print("-> Full backtest on the companion QC strategy project:")
print("   create_compile -> create_backtest -> read_backtest")
print()
print("Reference characteristics of the strategy (design expectations,")
print("NOT results measured here): CAGR ~8-12%, MaxDD ~30-40%, Sharpe ~0.5-0.8.")

MEASURED HERE (current snapshot, real data)
  Dow components screened: 30/30
  Dogs (top 10 by div yield): VZ, PFE, DOW, NKE, CVX, MCD, HD, IBM, PG, UNH
  Puppies (5 lowest-priced): DOW, PFE, NKE, VZ, PG

NOT MEASURED HERE: the 12-year annual-rebalance backtest
A faithful backtest needs the Dow composition AND each year's yield
as of each rebalance date -- historical index membership. Applying
TODAY's puppies to the past would be survivorship bias, so this
notebook does not fake it.

-> Full backtest on the companion QC strategy project:
   create_compile -> create_backtest -> read_backtest

Reference characteristics of the strategy (design expectations,
NOT results measured here): CAGR ~8-12%, MaxDD ~30-40%, Sharpe ~0.5-0.8.
